In [ ]:
📘 HÜCRE 1: Kurulum
Önce ortamı hazırlayalım. A100 GPU'nun aktif olduğundan emin olun.

Python

# @title 1. Kütüphane Kurulumu
# Gerekli modern kütüphaneleri yüklüyoruz.
!pip install git+https://github.com/amazon-science/chronos-forecasting.git
!pip install torch pandas numpy matplotlib mplfinance

import pandas as pd
import numpy as np
import torch
from chronos import ChronosPipeline
import mplfinance as mpf
import matplotlib.pyplot as plt
import tqdm

# GPU Kontrolü
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Kullanılan Cihaz: {device.upper()}")
if device == "cpu":
    print("⚠️ UYARI: GPU bulunamadı! İşlem çok yavaş olabilir. Lütfen 'Runtime > Change runtime type' menüsünden GPU seçin.")
📘 HÜCRE 2: Veri Yükleme (Sizin Formatınıza Özel)
Burada verinizi okuyoruz. Paylaştığınız görsele göre ilk sütun tarih, diğerleri küçük harfle open, high, low, close şeklinde.

Python

# @title 2. Veri Setini Yükleme
# Lütfen 'data.csv' dosyanızı Colab'in sol panelindeki dosyalar kısmına sürükleyip bırakın.

# Dosya adınız farklıysa burayı değiştirin
file_path = "data.csv"

try:
    # Görseldeki formata göre okuma yapıyoruz (index_col=0 tarih sütunu)
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)

    # Sütun isimlerindeki olası boşlukları temizle
    df.columns = df.columns.str.strip().str.lower()

    print("✅ Veri Başarıyla Yüklendi!")
    print(f"Toplam Satır: {len(df)}")
    print("Örnek Veri:")
    print(df[['open', 'high', 'low', 'close', 'volume']].tail())

    # Chronos için sadece 'close' verisi yeterlidir ama biz volatilite için diğerlerini de kullanacağız.
    historical_close = df['close']

except FileNotFoundError:
    print("❌ HATA: 'data.csv' dosyası bulunamadı. Lütfen dosyanızı yükleyin.")
    # Test için rastgele veri oluştur (Dosya yoksa kod patlamasın diye)
    dates = pd.date_range(start="2023-01-01", periods=1000, freq="h")
    df = pd.DataFrame({
        'open': np.random.rand(1000) + 1.0,
        'close': np.random.rand(1000) + 1.0,
        'high': np.random.rand(1000) + 1.1,
        'low': np.random.rand(1000) + 0.9,
        'volume': np.random.randint(100, 1000, 1000)
    }, index=dates)
    historical_close = df['close']
    print("⚠️ 'data.csv' bulunamadığı için rastgele test verisi oluşturuldu.")
📘 HÜCRE 3: Modeli Yükleme
Colab Pro+ gücünü burada kullanıyoruz.

Python

# @title 3. Amazon Chronos Modelini Yükle
# A100 GPU varsa 'large' modeli, yoksa 'small' modeli otomatik seçilir.

model_size = "amazon/chronos-t5-large" if device == "cuda" else "amazon/chronos-t5-small"

print(f"⏳ Model Yükleniyor ({model_size})...")

pipeline = ChronosPipeline.from_pretrained(
    model_size,
    device_map=device,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
)

print("✅ Model Hazır!")
📘 HÜCRE 4: Sentetik Mum Üretim Motoru
Burası projenin kalbi. Yapay zeka trendi (Close) belirler, matematiksel formülümüz ise mumun şeklini (Open, High, Low) oluşturur.

Python

# @title 4. Sentetik Veri Üretimi (Trend + Mum Giydirme)

def generate_candles(
    model,
    history_series,
    steps=168, # Kaç saatlik veri üretilsin? (Örn: 1 hafta = 168 saat)
    context_len=512 # Model geçmişe ne kadar baksın?
):
    print(f"🔄 Simülasyon Başlıyor: {steps} adet mum üretilecek...")

    # Veriyi Tensor'a çevir
    context_tensor = torch.tensor(history_series.values[-context_len:])

    generated_data = []
    last_close = history_series.iloc[-1]

    # Volatiliteyi hesapla (Sizin verideki 'atr' sütunu varsa kullanılabilir ama biz dinamik hesaplayalım)
    # Son 100 mumun oynaklığını baz alıyoruz.
    base_volatility = history_series.pct_change().std()

    # Parça parça üretim döngüsü (Chunk Loop)
    chunk_size = 24 # Her seferde 24 saat tahmin et

    for _ in tqdm.tqdm(range(0, steps, chunk_size)):

        # 1. AI TAHMİNİ (Sadece Close fiyatı)
        forecast = model.predict(
            context_tensor,
            prediction_length=chunk_size,
            num_samples=1
        )
        predicted_closes = forecast[0].mean(dim=0).cpu().numpy()

        # 2. MUM OLUŞTURMA (Post-Processing)
        for close_val in predicted_closes:
            # Mantık: Bir mumun açılışı, önceki mumun kapanışıdır.
            open_val = last_close

            # Mumun gövdesi
            body_max = max(open_val, close_val)
            body_min = min(open_val, close_val)

            # Fitil (Wick) Hesaplama:
            # Volatiliteye rastgelelik ekliyoruz (Log-Normal dağılım krizleri simüle eder)
            current_vol = open_val * base_volatility * np.random.lognormal(0, 0.4)

            high_val = body_max + (current_vol * np.random.rand())
            low_val = body_min - (current_vol * np.random.rand())

            # Hacim Simülasyonu (Fiyat sert değişirse hacim artar)
            price_change_pct = abs((close_val - open_val) / open_val)
            volume = int(10000 * (1 + (price_change_pct * 100)) * np.random.uniform(0.8, 1.2))

            generated_data.append({
                "open": open_val,
                "high": high_val,
                "low": low_val,
                "close": close_val,
                "volume": volume
            })

            last_close = close_val

        # 3. BAĞLAMI GÜNCELLE (Sliding Window)
        new_tensor = torch.tensor(predicted_closes)
        context_tensor = torch.cat((context_tensor, new_tensor))
        if len(context_tensor) > context_len:
            context_tensor = context_tensor[-context_len:]

    # DataFrame'e çevir
    syn_df = pd.DataFrame(generated_data)

    # Tarihleri oluştur (Gerçek verinin bittiği yerden devam et)
    last_date = history_series.index[-1]
    future_dates = pd.date_range(start=last_date, periods=len(syn_df)+1, freq='h')[1:]
    syn_df.index = future_dates
    syn_df.index.name = "datetime"

    return syn_df

# --- ÇALIŞTIR ---
# 500 Saatlik (Yaklaşık 20 gün) Gelecek Verisi Üret
synthetic_df = generate_candles(pipeline, historical_close, steps=500)

print("\n✅ Üretim Tamamlandı! İlk 5 satır:")
print(synthetic_df.head())
📘 HÜCRE 5: Görselleştirme
TradingView benzeri profesyonel bir grafik çizelim.

Python

# @title 5. Sonuçları Görselleştir (Grafik)
# Sütun isimlerini mplfinance formatına (Büyük harf) çevir
plot_df = synthetic_df.copy()
plot_df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# Grafik Teması
mc = mpf.make_marketcolors(up='#00ff00', down='#ff0000', inherit=True)
s = mpf.make_mpf_style(marketcolors=mc, style='nightclouds')

print("📊 Grafik Çiziliyor...")

mpf.plot(
    plot_df,
    type='candle',
    style=s,
    title='Yapay Zeka Tarafindan Uretilen Gelecek Senaryosu (Chronos AI)',
    ylabel='Fiyat',
    volume=True,
    figsize=(14, 7),
    mav=(20, 50) # 20 ve 50 mumluk hareketli ortalamaları da ekle
)
📘 HÜCRE 6: İndirme
Veriyi CSV olarak indir.

Python

# @title 6. Veriyi Kaydet ve İndir
from google.colab import files

filename = "sentetik_gelecek_verisi.csv"
synthetic_df.to_csv(filename)
files.download(filename)

print(f"📥 {filename} dosyası indiriliyor...")
💡 Nasıl Kullanılır?
Bu kodları sırasıyla çalıştırın.

Hücre 2'de sizden data.csv isteyecek. Sizin ekran görüntüsünü attığınız dosyayı oraya yükleyin.

Eğer dosya yüklemezseniz kod hata vermez, otomatik olarak rastgele test verisiyle çalışır (denemeniz için).

Çıktıyı Hücre 5'te grafik olarak göreceksiniz. Mumların şekli (fitiller, gövde) tamamen yapay zeka trendine ve matematiksel oynaklığa göre hesaplanmıştır.